Соединим воедино все ранее написанные куски и адаптируем их с учётом новых вводных. Реализуйте:
- функцию `train`, которая принимает конфиг обучения (уже в прекоде) и выполняет тренировку модели, сохраняя лучшую версию по метрикам на валидации;
- функцию `validate`, вызываемую внутри `train`, возвращающую accuracy текущей итерации на отложенной выборке;
- обновлённый код модели и загрузчика данных (базовая часть в прекоде) — с использованием параметров из конфига и заморозкой.

Класс мультимодального датасета `MultimodalDataset`, его `collate_fn` и необходимые трансформации, написанные нами в прошлых уроках, уже адаптированы и импортируются из модуля `dataset`.

In [ ]:
class Config:
    # Модели
    TEXT_MODEL_NAME = "bert-base-uncased"
    IMAGE_MODEL_NAME = "resnet50"
    
    # Какие слои размораживаем - совпадают с неймингом в моделях
    TEXT_MODEL_UNFREEZE = "encoder.layer.11|pooler"  
    IMAGE_MODEL_UNFREEZE = "layer.3|layer.4"  
    
    # Гиперпараметры
    BATCH_SIZE = 32
    TEXT_LR = 3e-5
    IMAGE_LR = 1e-4
    CLASSIFIER_LR = 5e-4
    EPOCHS = 10
    DROPOUT = 0.3
    HIDDEN_DIM = 256
    NUM_CLASSES = 5
    
    # Пути
    TRAIN_DF_PATH = "path/train.csv"
    VAL_DF_PATH = "path/val.csv"
    SAVE_PATH = "best_model.pth"

In [1]:
# `dataset` 
from functools import partial

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import numpy as np
import pandas as pd

from transformers import AutoTokenizer

import albumentations as A


class MultimodalDataset(Dataset):
    def __init__(self, config, transforms, ds_type="train"):
        if ds_type == "train":
            self.df = pd.read_csv(config.TRAIN_DF_PATH)
        else:
            self.df = pd.read_csv(config.VAL_DF_PATH)
        self.image_cfg = timm.get_pretrained_cfg(config.IMAGE_MODEL_NAME)
        self.tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "text"]
        label = self.df.loc[idx, "label"]

        img_path = self.df.loc[idx, "image_path"]
        image = Image.open(f"data/images/{img_path}").convert('RGB')
        image = self.transforms(image=np.array(image))["image"]

        return {"label": label, "image": image, "text": text}


def collate_fn(batch, tokenizer):
    texts = [item["text"] for item in batch]
    images = torch.stack([item["image"] for item in batch])
    labels = torch.LongTensor([item["label"] for item in batch])

    tokenized_input = tokenizer(texts,
                                return_tensors="pt",
                                padding="max_length",
                                truncation=True)

    return {
        "label": labels,
        "image": images,
        "input_ids": tokenized_input["input_ids"],
        "attention_mask": tokenized_input["attention_mask"]
    }


def get_transforms(config, ds_type="train"):
    cfg = timm.get_pretrained_cfg(config.IMAGE_MODEL_NAME)

    if ds_type == "train":
        transforms = A.Compose(
            [
                A.SmallestMaxSize(
                    max_size=max(cfg.input_size[1], cfg.input_size[2]), p=1.0),
                A.RandomCrop(
                    height=cfg.input_size[1], width=cfg.input_size[2], p=1.0),
                A.Affine(scale=(0.8, 1.2),
                        rotate=(-15, 15),
                        translate_percent=(-0.1, 0.1),
                        shear=(-10, 10),
                        fill=0,
                        p=0.8),
                A.CoarseDropout(num_holes_range=(2, 8),
                                hole_height_range=(int(0.07 * cfg.input_size[1]),
                                                int(0.15 * cfg.input_size[1])),
                                hole_width_range=(int(0.1 * cfg.input_size[2]),
                                                int(0.15 * cfg.input_size[2])),
                                fill=0,
                                p=0.5),
                A.ColorJitter(
                    brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.7),
                A.Normalize(mean=cfg.mean, std=cfg.std),
                A.ToTensorV2(p=1.0)
            ],
            seed=42,
        )
    else:
        transforms = A.Compose(
            [
                A.SmallestMaxSize(
                    max_size=max(cfg.input_size[1], cfg.input_size[2]), p=1.0),
                A.CenterCrop(
                    height=cfg.input_size[1], width=cfg.input_size[2], p=1.0),
                A.Normalize(mean=cfg.mean, std=cfg.std),
                A.ToTensorV2(p=1.0)
            ]
        )

    return transforms

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import re
import tqdm

import timm
import torch
import torch.nn as nn

from transformers import AutoModel, AutoTokenizer
from torch.optim import AdamW

# from dataset import MultimodalDataset, collate_fn, get_transforms


def set_requires_grad(module, unfreeze_pattern="", verbose=False):
    if len(unfreeze_pattern) == 0:
        for param, _ in module.named_parameters():
            param.requires_grad = False
        return

    pattern = re.compile(unfreeze_pattern)

    for name, param in module.named_parameters():
        if pattern.search(name):
            param.requires_grad = True
            if verbose:
                print(f"Разморожен слой: {name}")
        else:
            param.requires_grad = False


class MultimodalModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.text_model = AutoModel.from_pretrained(config.TEXT_MODEL_NAME)
        self.image_model = timm.create_model(
            config.IMAGE_MODEL_NAME,
            pretrained=True,
            num_classes=0
        )


        self.text_proj = nn.Linear(self.text_model.config.hidden_size, config.HIDDEN_DIM)
        self.image_proj = nn.Linear(self.image_model.num_features, config.HIDDEN_DIM)

        self.classifier = nn.Sequential(
            nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM // 2),
            nn.LayerNorm(config.HIDDEN_DIM // 2),
            nn.ReLU(),                           
            nn.Dropout(config.DROPOUT),                    
            nn.Linear(config.HIDDEN_DIM // 2, config.NUM_CLASSES),
        )

    def forward(self, input_ids, attention_mask, image):
        text_features = self.text_model(input_ids, attention_mask).last_hidden_state[:,  0, :]
        image_features = self.image_model(image)

        text_emb = self.text_proj(text_features)
        image_emb = self.image_proj(image_features)

        fused_emb = text_emb * image_emb
        
        logits = self.classifier(fused_emb)
        return logits


def train(config):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    best_acc = 0.
    
    # Инициализация модели
    model = MultimodalModel(config).to(device)
    tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
    # Оптимизатор 
    optimizer = AdamW([
        {'params': model.text_model.parameters(), 'lr': config.TEXT_LR},
        {'params': model.image_model.parameters(), 'lr': config.IMAGE_LR},
        {'params': model.classifier.parameters(), 'lr': config.CLASSIFIER_LR},
    ])
    
    criterion = nn.CrossEntropyLoss()
    
    # Загрузка данных
    transforms = get_transforms(config, ds_type="train")
    val_transforms = get_transforms(config, ds_type="val")

    train_dataset = MultimodalDataset(config, transforms)
    val_dataset = MultimodalDataset(config, val_transforms, ds_type="val")

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        collate_fn=partial(collate_fn, tokenizer=tokenizer)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        collate_fn=partial(collate_fn, tokenizer=tokenizer)
    )

    set_requires_grad(model.text_model, unfreeze_pattern=config.TEXT_MODEL_UNFREEZE, verbose=False)
    set_requires_grad(model.image_model, unfreeze_pattern=config.IMAGE_MODEL_UNFREEZE, verbose=False)
    
    # Цикл обучения
    for epoch in range(config.EPOCHS):
        model.train()
        for batch in tqdm(train_loader):
            inputs = {
                'input_ids': batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device),
                'image': batch['image'].to(device)
            }
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(**inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            optimizer.step()
            total_loss += loss.item()
        val_acc = validate(model, val_loader, device)
        print(f"Epoch {epoch+1}/{config.EPOCHS} | avg_Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), config.SAVE_PATH)


def validate(model, val_loader, device):
    model.eval()
    val_acc = 0.0
    with torch.no_grad():
        for batch in val_loader:
            inputs = {
                'input_ids': batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device),
                'image': batch['image'].to(device)
            }
            labels = batch["label"].to(device)

            outputs = model(**inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            val_acc += correct / total
    return val_acc



Обновите код функций `train/validate`, заменив ручной расчёт accuracy на f1-score из `torchmetrics`.

In [5]:
import torchmetrics


def train(config):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    best_f1 = 0
    
    # Инициализация модели
    model = MultimodalModel(config).to(device)
    tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
    # Оптимизатор 
    optimizer = AdamW([
        {'params': model.text_model.parameters(), 'lr': config.TEXT_LR},
        {'params': model.image_model.parameters(), 'lr': config.IMAGE_LR},
        {'params': model.classifier.parameters(), 'lr': config.CLASSIFIER_LR},
    ])
    
    criterion = nn.CrossEntropyLoss()
    
    # Загрузка данных
    transforms = get_transforms(config, ds_type="train")
    val_transforms = get_transforms(config, ds_type="val")

    train_dataset = MultimodalDataset(config, transforms)
    val_dataset = MultimodalDataset(config, val_transforms, ds_type="val")

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        collate_fn=partial(collate_fn, tokenizer=tokenizer)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        collate_fn=partial(collate_fn, tokenizer=tokenizer)
    )

    set_requires_grad(model.text_model, unfreeze_pattern=config.TEXT_MODEL_UNFREEZE, verbose=False)
    set_requires_grad(model.image_model, unfreeze_pattern=config.IMAGE_MODEL_UNFREEZE, verbose=False)
    
    F1_task = "binary" if config.NUM_CLASSES == 2 else "multiclass" 
    f1_metric = torchmetrics.F1Score(task=F1_task, num_classes=config.NUM_CLASSES).to(device) 

    # Цикл обучения
    for epoch in range(config.EPOCHS):
        model.train()
        f1_metric.reset()
        for batch in tqdm(train_loader):
            inputs = {
                'input_ids': batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device),
                'image': batch['image'].to(device)
            }
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(**inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            optimizer.step()
            score = f1_metric(preds=outputs, target=labels)
        train_f1 = f1_metric.compute()
        val_f1 = validate(model, val_loader, device, f1_metric)
        print(f"Epoch {epoch+1}/{config.EPOCHS} | train f1: {train_f1:.4f} | val f1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), config.SAVE_PATH)


# def validate(model, val_loader, device, f1_metric):
#     model.eval()
#     with torch.no_grad():
#         for batch in val_loader:
#             inputs = {
#                 'input_ids': batch['input_ids'].to(device),
#                 'attention_mask': batch['attention_mask'].to(device),
#                 'image': batch['image'].to(device)
#             }
#             labels = batch["label"].to(device)

#             outputs = model(**inputs)
#             score = f1_metric(preds=outputs, target=labels)
#         f1 = f1_metric.compute()
#     return f1

def validate(model, val_loader, device, f1_metric):
    model.eval()
    
    with torch.no_grad():
        for batch in val_loader:
            inputs = {
                'input_ids': batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device),
                'image': batch['image'].to(device)
            }
            labels = batch['label'].to(device)
            
            logits = model(**inputs)
            _, predicted = logits.argmax(dim=1)
            _ = f1_metric(preds=predicted, target=labels)
    
    return f1_metric.compute().cpu().numpy() 